<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week1/lab1/WPI_week1_lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biomedical Imaging: Deterministic Segmentation Baseline

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

**Clinical disclaimer:** The scores and flags in this lab are synthetic teaching outputs for workflow practice. They are not clinically validated measurements.


## Learning Objectives

By the end of this lab, you should be able to:

- Treat an image as structured numerical data.
- Normalize image intensities before threshold-based segmentation.
- Build a simple deterministic segmentation baseline using thresholding and morphology.
- Create a WPI-styled overlay for visual quality control.
- Convert a mask into a small structured output table.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week1_lab1_responses_LastName_FirstName.docx`

Write Q1-Q5 in the Word document, using 2-5 sentences per question. Keep longer written responses in the Word document rather than in notebook markdown cells. The notebook should contain the generated plots, tables, and short notes requested in each part.


## Workflow

This lab follows a classic baseline pipeline:

`Image -> Normalize -> Threshold -> Morphology -> Overlay -> QC -> Structured Output`

A deterministic baseline is not an AI model. It is a transparent reference workflow that helps you inspect the data, debug assumptions, and understand what an automated method would need to improve.


## Setup

Run this setup cell first. The notebook installs the small set of packages used in the lab, clones the public course helper repo in Colab, and imports the shared data/style helpers.



In [ ]:
#@title Setup course environment
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scikit-image",
    "pandas",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    repo_dir = Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])

sys.path.insert(0, str(repo_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import data as sk_data
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.morphology import closing, disk, opening

from wpi_ai_bootcamp.data import load_imaging_sample
from wpi_ai_bootcamp.notebook import setup_lab, make_wpi_overlay

WPI_COLORS = setup_lab()
print("Setup complete.")



## Data Loading

This lab does not use a committed image file. The image is loaded at runtime from `scikit-image` sample data.

Primary path: `wpi_ai_bootcamp.data.load_imaging_sample("retina")`, which calls `skimage.data.retina()` and returns source metadata.

Fallback path: direct `skimage.data.retina()`. If that sample is unavailable, the notebook uses `skimage.data.immunohistochemistry()` so the lab can still run in Colab.



In [ ]:
try:
    from wpi_ai_bootcamp.data import load_imaging_sample
    image_rgb, source = load_imaging_sample("retina")
    source_name = source.name
    source_url = source.url
except Exception as exc:
    print("Course loader unavailable; using skimage.data directly.")
    print(type(exc).__name__, str(exc)[:120])
    try:
        image_rgb = sk_data.retina()
        source_name = "scikit-image retina sample image"
        source_url = "https://scikit-image.org/docs/stable/api/skimage.data.html"
    except Exception as retina_exc:
        print("Retina sample unavailable; using immunohistochemistry sample.")
        print(type(retina_exc).__name__, str(retina_exc)[:120])
        image_rgb = sk_data.immunohistochemistry()
        source_name = "scikit-image immunohistochemistry sample image"
        source_url = "https://scikit-image.org/docs/stable/api/skimage.data.html"

print(source_name)
print(source_url)
print("shape:", image_rgb.shape, "dtype:", image_rgb.dtype, "range:", image_rgb.min(), image_rgb.max())


## Part 1 — Image Inspection And Baseline Normalization

Biomedical images are arrays. Before segmenting anything, check shape, type, intensity range, and whether the image is grayscale or multi-channel.


In [ ]:
if image_rgb.ndim == 3:
    image_gray = rgb2gray(image_rgb)
else:
    image_gray = image_rgb.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image_rgb)
axes[0].set_title("Original image")
axes[0].axis("off")
axes[1].imshow(image_gray, cmap="gray")
axes[1].set_title("Grayscale image")
axes[1].axis("off")
plt.show()

print("grayscale shape:", image_gray.shape)
print("grayscale dtype:", image_gray.dtype)
print("grayscale range:", float(np.min(image_gray)), float(np.max(image_gray)))


## Normalize Intensities

Thresholding depends on intensity values. Normalization maps values to a common range so a threshold is easier to reason about. Percentile normalization is often more robust when a few pixels are unusually bright or dark.


In [ ]:
def normalize_to_01(image, mode="percentile", low_percentile=1, high_percentile=99):
    image = image.astype(np.float32)
    if mode == "minmax":
        lo = float(np.min(image))
        hi = float(np.max(image))
    elif mode == "percentile":
        lo = float(np.percentile(image, low_percentile))
        hi = float(np.percentile(image, high_percentile))
    else:
        raise ValueError("mode must be 'percentile' or 'minmax'")

    if hi <= lo:
        raise ValueError("normalization range is empty")
    return np.clip((image - lo) / (hi - lo), 0, 1)

NORM_MODE = "percentile"
LOW_PERCENTILE = 1
HIGH_PERCENTILE = 99

image01 = normalize_to_01(image_gray, NORM_MODE, LOW_PERCENTILE, HIGH_PERCENTILE)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image01, cmap="gray")
axes[0].set_title("Normalized image")
axes[0].axis("off")
axes[1].hist(image01.ravel(), bins=50, color=WPI_COLORS["crimson"])
axes[1].set_title("Normalized intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
plt.show()

print("normalized range:", float(image01.min()), float(image01.max()))


### Part 1 Assessment — Image Inspection And Baseline Normalization (20 pts)

- Notebook artifact: display the original image, grayscale image, normalized image, and normalized intensity histogram. **12 pts**
- Word response Q1: Why do we normalize intensities before thresholding? **8 pts**
- Grading criteria: the notebook output should show the image shape/range and a valid normalized range near `[0, 1]`; the Word response should connect normalization to comparable intensity scale and threshold behavior.


### Part 2 Assessment — Normalization Setting Comparison (20 pts)

- Notebook artifact: rerun normalization after changing `NORM_MODE`, `LOW_PERCENTILE`, or `HIGH_PERCENTILE`, and keep the comparison visible in the notebook. **12 pts**
- Word response Q2: Compare normalization settings and describe what changed in the image or histogram. **8 pts**
- Grading criteria: the notebook should show evidence of at least one parameter change; the Word response should describe a concrete visual or histogram difference and why it matters before thresholding.


## Part 3 — Thresholding

A threshold converts continuous intensities into a binary mask. Otsu thresholding chooses a cutoff from the histogram. Manual thresholding is useful when you want to inspect how a specific cutoff changes the mask.


In [ ]:
THRESH_METHOD = "otsu"  # "otsu" or "manual"
MANUAL_THRESH = 0.50
INVERT_MASK = False

if THRESH_METHOD == "otsu":
    threshold_value = float(threshold_otsu(image01))
elif THRESH_METHOD == "manual":
    threshold_value = float(MANUAL_THRESH)
else:
    raise ValueError("THRESH_METHOD must be 'otsu' or 'manual'")

mask = image01 >= threshold_value
if INVERT_MASK:
    mask = ~mask

mask_fraction = float(mask.mean())
print("threshold:", threshold_value)
print("mask fraction:", mask_fraction)
if mask_fraction == 0 or mask_fraction == 1:
    print("QC warning: mask is empty or full. Adjust thresholding or inversion.")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image01, cmap="gray")
axes[0].set_title("Normalized image")
axes[0].axis("off")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"Mask ({THRESH_METHOD})")
axes[1].axis("off")
plt.show()


### Part 3 Assessment — Thresholding (20 pts)

- Notebook artifact: run Otsu thresholding and at least one manual threshold between `0.2` and `0.8`; keep the mask outputs visible. **12 pts**
- Word response Q3: What does Otsu thresholding do conceptually? **8 pts**
- Grading criteria: the notebook should show a threshold value and non-empty mask output; the Word response should mention histogram separation and the idea of choosing a cutoff that separates intensity groups.


## Part 4 — Morphological Cleanup

Morphology edits mask shape. Opening can remove small isolated regions. Closing can fill small gaps. The radius controls how aggressive the cleanup is.


In [ ]:
MORPH_RADIUS = 3

selem = disk(MORPH_RADIUS)
mask_open = opening(mask, selem)
mask_clean = closing(mask_open, selem)

clean_fraction = float(mask_clean.mean())
print("clean mask fraction:", clean_fraction)
if clean_fraction == 0 or clean_fraction == 1:
    print("QC warning: cleanup created an empty or full mask. Try a smaller radius or a different threshold.")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mask, cmap="gray")
axes[0].set_title("Threshold mask")
axes[0].axis("off")
axes[1].imshow(mask_open, cmap="gray")
axes[1].set_title("After opening")
axes[1].axis("off")
axes[2].imshow(mask_clean, cmap="gray")
axes[2].set_title("After closing")
axes[2].axis("off")
plt.show()


### Part 4 Assessment — Morphology Sweep (20 pts)

- Notebook artifact: show masks for radius `1`, `3`, and `5`, plus the `radius_table`. **12 pts**
- Word response Q4: What is the difference between opening and closing, and how did radius affect your mask? **8 pts**
- Grading criteria: the notebook should show a visible sweep across radii; the Word response should distinguish opening from closing and explain how larger radius changes mask cleanup.


In [ ]:
radius_results = []
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, radius in zip(axes, [1, 3, 5]):
    cleaned = closing(opening(mask, disk(radius)), disk(radius))
    roi_ratio = float(cleaned.mean())
    radius_results.append({"radius": radius, "roi_ratio": roi_ratio})
    ax.imshow(cleaned, cmap="gray")
    ax.set_title(f"radius={radius}\nroi={roi_ratio:.3f}")
    ax.axis("off")
plt.show()

radius_table = pd.DataFrame(radius_results)
radius_table


## Part 5 — Overlay, QC, And Structured Output

An overlay helps you check whether the mask highlights a meaningful region. The mask is shown in WPI Crimson so the segmented region is visually distinct from the grayscale context.


In [ ]:
overlay = make_wpi_overlay(image01, mask_clean)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(image01, cmap="gray")
axes[0].set_title("Image")
axes[0].axis("off")
axes[1].imshow(mask_clean, cmap="gray")
axes[1].set_title("Clean mask")
axes[1].axis("off")
axes[2].imshow(overlay)
axes[2].set_title("Overlay")
axes[2].axis("off")
plt.show()


### TODO: Visual QC Checkpoint

Inspect the overlay. In 2-3 sentences, describe whether the mask captures a coherent region, misses important structure, or includes too much background. Name one parameter you would adjust next.


## Structured Output

A useful workflow produces more than a picture. Here we convert the mask into a small table with fixed columns. These values are synthetic teaching outputs and should not be interpreted as clinical measurements.


In [ ]:
roi_ratio = float(mask_clean.mean())
risk_score = float(np.clip(roi_ratio / 0.20, 0, 1))
alarm = int(risk_score > 0.50)
quality = int(0.01 < roi_ratio < 0.60)
review_flag = int(quality == 0)

output_table = pd.DataFrame([
    {
        "roi_ratio": roi_ratio,
        "risk_score": risk_score,
        "alarm": alarm,
        "quality": quality,
        "review_flag": review_flag,
    }
])
output_table


## Mini Parameter Experiment

Now compare one parameter across several values. The goal is not to optimize the pipeline. The goal is to show that deterministic choices change downstream outputs.


In [ ]:
experiment_rows = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, radius in zip(axes, [1, 3, 5]):
    candidate_mask = closing(opening(mask, disk(radius)), disk(radius))
    candidate_overlay = make_wpi_overlay(image01, candidate_mask)
    candidate_roi = float(candidate_mask.mean())
    candidate_risk = float(np.clip(candidate_roi / 0.20, 0, 1))
    experiment_rows.append({
        "radius": radius,
        "roi_ratio": candidate_roi,
        "risk_score": candidate_risk,
        "quality": int(0.01 < candidate_roi < 0.60),
        "review_flag": int(not (0.01 < candidate_roi < 0.60)),
    })
    ax.imshow(candidate_overlay)
    ax.set_title(f"radius={radius}")
    ax.axis("off")
plt.show()

experiment_table = pd.DataFrame(experiment_rows)
experiment_table


### Part 5 Assessment — Final Parameter Decision (20 pts)

- Notebook artifact: show the WPI Crimson overlay, the structured output table, mini experiment overlays, and the experiment table. **12 pts**
- Word response Q5: Which final parameter setting would you keep, based on overlay, QC, and structured output table? **8 pts**
- Grading criteria: the notebook should include the required schema columns and at least three radius comparisons; the Word response should use both visual evidence and table values such as `roi_ratio` or `risk_score`.


## Word Response Prompts

Write responses to Q1-Q5 in `WPI_week1_lab1_responses_LastName_FirstName.docx`. Use 2-5 sentences per question.

1. Why do we normalize intensities before thresholding?
2. Compare normalization settings and describe what changed.
3. What does Otsu thresholding do conceptually?
4. What is the difference between opening and closing, and how did radius affect your mask?
5. Which final parameter setting would you keep, based on overlay, QC, and structured output table?


## Optional Challenge

Try a different `skimage.data` image, such as `immunohistochemistry`, and rerun the same pipeline. Which steps transfer cleanly, and which parameter choices need to change?


## Attribution

- Data: scikit-image sample data, loaded through `skimage.data`.
- Libraries: NumPy, pandas, Matplotlib, and scikit-image.
- WPI colors: WPI Institutional Guidelines, Crimson `#AC2B37` and Gray `#A9B0B7`.
